In [ ]:
import os
import numpy as np
import scipy.io
from scipy.stats import kurtosis, skew
from scipy.fft import fft
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# === 설정 ===
window_size = 2048
step_size = 2048
label_map = {
    "B007": "Ball_007", "B014": "Ball_014", "B021": "Ball_021",
    "IR007": "InnerRace_007", "IR014": "InnerRace_014", "IR021": "InnerRace_021",
    "OR007": "OuterRace_007", "OR014": "OuterRace_014", "OR021": "OuterRace_021",
    "Time_Normal": "Normal"
}

# === 데이터 로드 및 윈도우 분할 ===
def extract_signal_and_label(file_path):
    mat = scipy.io.loadmat(file_path)
    fname = os.path.basename(file_path)
    name = fname.split(".")[0]
    de_key = [k for k in mat.keys() if "DE_time" in k][0]
    signal = mat[de_key].squeeze()
    label = next((label_map[k] for k in label_map if k in name), "Unknown")
    samples, labels = [], []
    for i in range(0, len(signal) - window_size + 1, step_size):
        samples.append(signal[i:i + window_size])
        labels.append(label)
    return samples, labels

# === 피처 추출 ===
def extract_features(signal):
    features = [
        np.mean(signal), np.std(signal), np.sqrt(np.mean(signal**2)),
        np.max(signal), np.min(signal), np.ptp(signal),
        skew(signal), kurtosis(signal),
        np.max(np.abs(signal)) / np.sqrt(np.mean(signal**2))
    ]
    fft_vals = np.abs(fft(signal))[:len(signal)//2]
    spectral_centroid = np.sum(np.arange(len(fft_vals)) * fft_vals) / np.sum(fft_vals)
    spectral_bandwidth = np.sqrt(np.sum(((np.arange(len(fft_vals)) - spectral_centroid) ** 2) * fft_vals) / np.sum(fft_vals))
    features += [np.mean(fft_vals), np.std(fft_vals), spectral_centroid, spectral_bandwidth]
    return features

# === 현실 노이즈 시뮬레이션 ===
def generate_realistic_noise(length, kind="vibration"):
    np.random.seed(42)
    if kind == "vibration":
        t = np.linspace(0, 1, length)
        vibration = 0.3 * np.sin(2 * np.pi * 60 * t)
        noise = 0.05 * np.random.normal(0, 1, length)
        return vibration + noise
    elif kind == "irregular_impact":
        base = 0.02 * np.random.normal(0, 1, length)
        for _ in range(10):
            idx = np.random.randint(0, length - 20)
            base[idx:idx + 20] += np.hanning(20) * np.random.uniform(0.5, 1.0)
        return base
    return np.random.normal(0, 1, length)

# === 전체 실행 파이프라인 ===
def main(data_dir):
    file_list = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".mat")])
    all_samples, all_labels = [], []
    for file_path in file_list:
        s, l = extract_signal_and_label(file_path)
        all_samples.extend(s)
        all_labels.extend(l)

    X = np.array(all_samples)
    y = np.array(all_labels)

    X_features = np.array([extract_features(s) for s in X])
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_features)

    # 원본 RF 훈련
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_scaled, y_encoded)

    # 현실 노이즈 추가 → 재학습
    realistic_noisy_features = []
    for signal in X:
        vib = generate_realistic_noise(len(signal), "vibration")
        imp = generate_realistic_noise(len(signal), "irregular_impact")
        mixed = signal + vib + imp
        realistic_noisy_features.append(extract_features(mixed))

    X_real = np.array(realistic_noisy_features)
    y_real = le.transform(y)
    X_real_scaled = scaler.transform(X_real)

    # 원본 + 현실 노이즈로 재학습
    X_aug = np.vstack([X_features, X_real])
    y_aug = np.concatenate([y_encoded, y_real])
    X_aug_scaled = scaler.fit_transform(X_aug)

    X_train, X_test, y_train, y_test = train_test_split(
        X_aug_scaled, y_aug, test_size=0.2, stratify=y_aug, random_state=42)

    clf_aug = RandomForestClassifier(n_estimators=100, random_state=42)
    clf_aug.fit(X_train, y_train)

    y_pred_real = clf_aug.predict(X_real_scaled)
    acc = accuracy_score(y_real, y_pred_real)
    f1 = f1_score(y_real, y_pred_real, average="macro")
    cm = confusion_matrix(y_real, y_pred_real)

    # 시각화
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title("Confusion Matrix (Reality Noise Retrained)")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

# 사용 예시:
# if __name__ == "__main__":
#     main("./data_cwru/raw")